In [1]:
import pandas as pd
import numpy as np

CURRENT_YEAR = 2026

# ============================================================
# LOAD FILES
# ============================================================

df = pd.read_csv("data/chevrolet.csv")
dep_df = pd.read_csv("data/annual_dep_rate.csv")

# ============================================================
# CLEAN DEPRECIATION DATA
# ============================================================

dep_df["make"] = dep_df["make"].astype(str).str.strip().str.upper()
dep_df["model"] = dep_df["model"].astype(str).str.strip().str.upper()

# ============================================================
# CHEVROLET MODEL MAPPING
# ============================================================

MODEL_MAPPING = {
    "chevrolet-blazer": "BLAZER",
    "chevrolet-camaro": "CAMARO",
    "chevrolet-captiva": "CAPTIVA",
    "chevrolet-colorado": "COLORADO",
    "chevrolet-corvette": "CORVETTE",
    "chevrolet-cruze": "CRUZE",
    "chevrolet-equinox": "EQUINOX",
    "chevrolet-groove": "GROOVE",
    "chevrolet-impala": "IMPALA",
    "chevrolet-malibu": "MALIBU",
    "chevrolet-silverado": "SILVERADO",
    "chevrolet-suburban": "SUBURBAN",
    "chevrolet-tahoe": "TAHOE",
    "chevrolet-trailblazer": "TRAILBLAZER",
    "chevrolet-traverse": "TRAVERSE",
    "chevrolet-trax": "TRAX",
}

df["make"] = "CHEVROLET"

df["model_name"] = (
    df["model_slug"]
    .map(MODEL_MAPPING)
    .astype(str)
    .str.upper()
)

# ============================================================
# BUILD LOOKUP
# ============================================================

dep_lookup = (
    dep_df.groupby(["make", "model"])["annual_dep_rate"]
    .mean()
    .reset_index()
)

# ============================================================
# MERGE DEPRECIATION RATE
# ============================================================

df = df.merge(
    dep_lookup,
    left_on=["make", "model_name"],
    right_on=["make", "model"],
    how="left"
)

# ============================================================
# CAR AGE
# ============================================================

df["car_age"] = CURRENT_YEAR - df["year"]
df["car_age"] = df["car_age"].clip(lower=0)

# ============================================================
# DEPRECIATED VALUE
# ============================================================

def calc_depreciated_value(row):

    rate = row["annual_dep_rate"]

    if pd.isna(rate):
        return np.nan

    age = row["car_age"]

    if age == 0:
        return round(row["price_avg_aed"], 0)

    return round(
        row["price_avg_aed"] * ((1 - rate) ** age),
        0
    )

df["depreciated_value"] = df.apply(
    calc_depreciated_value,
    axis=1
)

# ============================================================
# VALIDATION
# ============================================================

print("Total Rows:", len(df))
print("Matched Rates:", df["annual_dep_rate"].notna().sum())
print("Missing Rates:", df["annual_dep_rate"].isna().sum())

print("\nUnmatched Models:")
print(
    df[df["annual_dep_rate"].isna()]
    ["model_slug"]
    .drop_duplicates()
    .tolist()
)

# ============================================================
# SAVE
# ============================================================

df.to_csv(
    "data/chevrolet_with_depreciation_annual_rate.csv",
    index=False
)

print("\nSaved: data/chevrolet_with_depreciation_annual_rate.csv")

Total Rows: 985
Matched Rates: 691
Missing Rates: 294

Unmatched Models:
['chevrolet-avalanche', 'chevrolet-aveo', 'chevrolet-aveo-hatchback', 'chevrolet-aveo5', 'chevrolet-bolt', 'chevrolet-bolt-euv', 'chevrolet-camaro-convertible', 'chevrolet-camaro-zl1', 'chevrolet-caprice', 'chevrolet-captiva-ev', 'chevrolet-captiva-phev', 'chevrolet-cruze-hatchback', 'chevrolet-csv-cr8', 'chevrolet-epica', 'chevrolet-equinox-ev', 'chevrolet-express', 'chevrolet-lumina', 'chevrolet-optra', 'chevrolet-sonic', 'chevrolet-sonic-hatchback', 'chevrolet-spark', 'chevrolet-spark-euv', 'chevrolet-t-series', 'chevrolet-uplander']

Saved: data/chevrolet_with_depreciation_annual_rate.csv
